# Oil and Gas Anomaly Detection — Section 1a: Dataset Preparation

**Project:** *Oil and Gas Anomaly Detection Using Feature Engineering and Machine Learning*

This notebook converts the raw 3W dataset (version 2.0.0, Parquet files organised in one folder per class, `0`–`9`) into tabular, feature-engineered datasets.

**Pipeline implemented in this notebook**

1. **Variable selection** — 8 process variables are kept: `P-PDG`, `P-TPT`, `T-TPT`, `P-MON-CKP`, `T-JUS-CKP`, `P-JUS-CKGL`, `T-MON-CKP`, `QGL`, plus `timestamp` and `class`.
2. **Label preparation** — transient labels `100 + k` are mapped to the steady-state label `k` for classes `{1, 2, 5, 6, 7, 8, 9}`; only the rows whose label equals the folder's class are kept.
3. **Window-based segmentation** — the time series is split into consecutive, non-overlapping **30-second windows** ("instances").
4. **Missing values** — missing sensor values are replaced by `0`; missing feature values are also replaced by `0`.
5. **Statistical feature extraction** — 16 statistical features per variable and per window (8 × 16 = **128 features**).
6. **Dataset assembly** — for every anomaly class `k`, its windows are concatenated with the normal-operation windows (class `0`) and saved as `final_data0_k.csv`; the class-0 windows alone are saved as `final_data0.csv`.

> The CSV files generated here are the inputs of the three experiments (notebooks `02`–`04`).
> The class-9 file used by the experiments is regenerated in notebook `01b_dataset_preparation_class9.ipynb`.

*Documentation (markdown cells and `#` comment lines) was added for publication; the original executable code and its outputs are unchanged.*

## 1. Environment setup

In [1]:
# ============================================================
# 1. ENVIRONMENT SETUP
# ============================================================
# Import data-handling libraries. The scikit-learn, XGBoost and Optuna
# imports are not used in this notebook; they are kept unchanged.
# RuntimeWarnings (e.g. from statistics on constant windows) are silenced.
import pandas as pd 
import numpy as np
import os
import glob

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score

from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score , GroupKFold, GroupShuffleSplit,StratifiedGroupKFold
import optuna
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

## 2. Feature engineering: statistical features per window

`extract_features` receives the rows of **one 30-second window** and returns one feature vector. For each of the 8 variables it computes 16 statistics:

| Group | Features |
|---|---|
| Location | mean, median, mean of absolute values |
| Dispersion | standard deviation, min, max, range, 25th / 75th percentiles, IQR, mean absolute deviation |
| Energy | root mean square (RMS), signal energy (sum of squares) |
| Shape | skewness, kurtosis |
| Sign changes | zero-crossing count |

In [2]:
# ============================================================
# 2. FEATURE ENGINEERING - STATISTICAL FEATURE EXTRACTION
# ============================================================
# Compute 16 whole-window statistical features for each of the 8 process
# variables of one window (group). Output: a pandas Series of 128 features
# named '<variable>_<statistic>'.
# Note: `diffs` (first differences) is computed but not used in the features.
def extract_features(group: pd.DataFrame) -> pd.Series:
    feats = {}
    group = group.copy()
    variables = ['P-PDG',
        'P-TPT',
        'T-TPT',
        'P-MON-CKP',
        'T-JUS-CKP',
        'P-JUS-CKGL',
        'T-MON-CKP',
        'QGL'
    ]

    # Whole-sequence statistical features
    for col in variables:
        data = group[col]
        diffs = data.diff().fillna(0)

        # Basic stats
        feats[f'{col}_mean']       = data.mean()
        feats[f'{col}_std']        = data.std()
        feats[f'{col}_min']        = data.min()
        feats[f'{col}_max']        = data.max()
        feats[f'{col}_median']     = data.median()
        feats[f'{col}_range']      = data.max() - data.min()
        feats[f'{col}_q25']        = data.quantile(0.25)
        feats[f'{col}_q75']        = data.quantile(0.75)
        feats[f'{col}_iqr']        = data.quantile(0.75) - data.quantile(0.25)
        feats[f'{col}_mad']        = (data - data.mean()).abs().mean() # mean absolute deviation
        feats[f'{col}_mean_abs']   = data.abs().mean()
        feats[f'{col}_rms']        = np.sqrt((data**2).mean())      # root mean square
        feats[f'{col}_energy']     = (data**2).sum()                # signal energy

        # Optional: skewness & kurtosis (requires pandas >= 1.1 or scipy)
        feats[f'{col}_skew']       = data.skew()
        feats[f'{col}_kurtosis']   = data.kurtosis()

        # Zero‐crossing count (how many times signal changes sign)
        feats[f'{col}_zero_crossings'] = ((data.shift(1) * data) < 0).sum()
    return pd.Series(feats)

## 3. Data loading, cleaning, labelling and window-based segmentation

`features_engineering(folder, filename)` processes **one Parquet file** of the 3W dataset:

1. Reads the file and moves the `timestamp` index into a column.
2. Keeps the 8 process variables, `timestamp` and `class`.
3. For transient-capable classes `{1, 2, 5, 6, 7, 8, 9}`, relabels the transient label `100 + k` as `k` (the class number is read from the folder name).
4. Keeps only the rows labelled with the folder's class `k` (for folder `0`, the normal-operation rows).
5. Assigns each row to a 30-second window: `instance = floor((t - t_first) / 30 s) + 1`, named `class_<k>_<n>`.
6. Replaces missing sensor values by `0`, extracts the statistical features per window (`extract_features`), replaces missing feature values by `0`, and attaches the window label.

In [3]:
# ============================================================
# 3. DATA LOADING, LABELLING AND 30-SECOND WINDOW SEGMENTATION
# ============================================================
# Process one Parquet file: select variables, normalise transient labels,
# keep the rows of the folder's class, segment them into consecutive
# non-overlapping 30-second windows, impute missing values with 0 and
# return one row of statistical features (+ label) per window.
def features_engineering(folder: str, filename: str) -> pd.DataFrame:
    
    # read
    path = os.path.join(folder, filename)
    data = pd.read_parquet(path, engine="pyarrow")

    data.reset_index(inplace = True )
    
    variables = [
        'P-PDG','P-TPT','T-TPT','P-MON-CKP','T-JUS-CKP','P-JUS-CKGL','T-MON-CKP','QGL'
    ]
    cols = ['timestamp'] + variables + ['class']
    data = data[cols].copy()
    
    # Fill NaN class values
    data['class'] = data['class'].dropna()

    # transient normalization (map 100+k -> k)
    transient_files = {1,2,5,6,7,8,9}
    
    num_of_transient = int(os.path.basename(folder))

    if num_of_transient in transient_files :
        
        mask = data['class'] == (100 + num_of_transient)
        data.loc[mask, 'class'] = num_of_transient
        
    # data_0 = data.loc[data['class'] == 0].copy()
        
    # # instance id: 1,2,3... every 30s from the first timestamp
    # t0 = data_0["timestamp"].min()
    # instance_numbers_0 = ((data_0["timestamp"] - t0).dt.total_seconds() // 30).astype(int) + 1
    # data_0["instance"] =  "class_0_" + instance_numbers_0.astype(str)

  
    data_t = data.loc[data['class'] == num_of_transient].copy()
    
    # instance id: 1,2,3... every 30s from the first timestamp
    t1 = data_t['timestamp'].min()
    instance_numbers_t = ((data_t['timestamp'] - t1).dt.total_seconds() // 30).astype('Int64') + 1
    data_t['instance'] = f"class_{num_of_transient}_" + instance_numbers_t.astype(str)

    data  = data_t.copy()

    data.drop(columns = ['timestamp'],inplace = True)
    data[variables]=data[variables].fillna(0)
    feature_data = data.groupby('instance').apply(extract_features,include_groups=False).fillna(0).reset_index()
    # 2. Add label per sequence
    label_data = data.groupby('instance')['class'].first().reset_index()
    feature_data = feature_data.merge(label_data, on='instance', how='left')
    return feature_data

In [4]:
# Earlier version of `features_engineering`, kept fully commented out
# in the original notebook (not executed).
# def features_engineering(folder: str, filename: str) -> pd.DataFrame:
    
#     # read
#     path = os.path.join(folder, filename)
#     data = pd.read_parquet(path, engine="pyarrow")

#     data.reset_index(inplace = True )
    
#     variables = [
#         'P-PDG','P-TPT','T-TPT','P-MON-CKP','T-JUS-CKP','P-JUS-CKGL','T-MON-CKP','QGL'
#     ]
#     cols = ['timestamp'] + variables + ['class']
#     data = data[cols].copy()
    
#     # Fill NaN class values
#     data['class'] = data['class'].dropna()

#     # transient normalization (map 100+k -> k)
#     transient_files = {1,2,5,6,7,8,9}
    
#     num_of_transient = int(os.path.basename(folder))

#     if num_of_transient in transient_files :
        
#         mask = data['class'] == (100 + num_of_transient)
#         data.loc[mask, 'class'] = num_of_transient
        
  
#     data_t = data.loc[data['class'] == num_of_transient].copy()
    
#     # instance id: 1,2,3... every 30s from the first timestamp
#     t1 = data_t['timestamp'].min()
#     instance_numbers_t = ((data_t['timestamp'] - t1).dt.total_seconds() // 30).astype('Int64') + 1
#     data_t['instance'] = f"class_{num_of_transient}_" + instance_numbers_t.astype(str)

#     data  = data_t.copy()

#     data.drop(columns = ['timestamp'],inplace = True)
#     data[variables]=data[variables].fillna(0)
#     feature_data = data.groupby('instance').apply(extract_features,include_groups=False).fillna(0).reset_index()
#     # 2. Add label per sequence
#     label_data = data.groupby('instance')['class'].first().reset_index()
#     feature_data = feature_data.merge(label_data, on='instance', how='left')
#     return feature_data


## 4. Per-class dataset generation

For each class folder the same procedure is applied:

* list the Parquet files of the folder and process the first *N* files with `features_engineering`;
* for anomaly classes, remove any window labelled `0`;
* concatenate the class-`k` windows with the class-`0` windows (`final_data0`), drop the `instance` identifier and save `final_data0_k.csv`.

| Folder (class) | Files processed (`file_names[:N]`) | Output CSV |
|---|---|---|
| 0 | 30 | `final_data0.csv` (keeps the `instance` column) |
| 1 | 15 | `final_data0_1.csv` |
| 2 | 40 | `final_data0_2.csv` |
| 3 | 30 | `final_data0_3.csv` |
| 4 | 120 | `final_data0_4.csv` |
| 5 | 50 | `final_data0_5.csv` |
| 6 | 40 | `final_data0_6.csv` |
| 7 | 8 | `final_data0_7.csv` |
| 8 | 20 | `final_data0_8.csv` |
| 9 | 7 | `final_data0_9.csv` (superseded by notebook `01b`) |

> `os.listdir` does not guarantee a file order, so the subset selected by `file_names[:N]` depends on the file system.

# Folder 0

In [17]:
# ============================================================
# 4.0 CLASS 0 - NORMAL OPERATION
# ============================================================
# List the Parquet files of folder 0 (normal operation).
folder = '/kaggle/input/3w-dataset/2.0.0/0'
file_names = os.listdir(folder)

In [18]:
# Extract windowed statistical features from the first 30 files and concatenate them.
rows = []
for i  in file_names[:30]: 
    data = features_engineering(folder,i)
    rows.append(data)
    
final_data0= pd.concat(rows)

In [19]:
# Display the class-0 feature table with a fresh index (display only; not assigned).
final_data0.reset_index(drop= True)

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,class_0_1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,class_0_10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,class_0_100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,class_0_101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,class_0_102,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16909,class_0_95,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
16910,class_0_96,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
16911,class_0_97,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
16912,class_0_98,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [20]:
# Display the class-0 feature table.
final_data0

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,class_0_1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,class_0_10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,class_0_100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,class_0_101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,class_0_102,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
592,class_0_95,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
593,class_0_96,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
594,class_0_97,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
595,class_0_98,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [21]:
# Export the class-0 feature dataset (with the `instance` column).
# Save to CSV
final_data0.to_csv('final_data0.csv', index=False)

# Folder 1

In [22]:
# ============================================================
# 4.1 CLASS 1
# ============================================================
# List the Parquet files of folder 1.
folder = '/kaggle/input/3w-dataset/2.0.0/1'
file_names = os.listdir(folder)

In [23]:
# Extract windowed statistical features from the first 15 files of folder 1 and concatenate them.
rows = []
for i  in file_names[:15] : 
    data = features_engineering(folder,i)
    rows.append(data)
    
final_data1= pd.concat(rows)

In [24]:
# Remove windows labelled as normal operation (class 0) from the class-1 table.
final_data1 = final_data1[final_data1['class']!=0]

In [25]:
# Reset the row index after filtering.
final_data1.reset_index(drop= True, inplace = True)

In [26]:
# Display the class-1 feature table.
final_data1

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,class_1_1,-1.180116e+42,4.721636e+26,-1.180116e+42,-1.180116e+42,-1.180116e+42,0.0,-1.180116e+42,-1.180116e+42,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,class_1_10,-1.180116e+42,4.721636e+26,-1.180116e+42,-1.180116e+42,-1.180116e+42,0.0,-1.180116e+42,-1.180116e+42,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,class_1_100,-1.180116e+42,4.721636e+26,-1.180116e+42,-1.180116e+42,-1.180116e+42,0.0,-1.180116e+42,-1.180116e+42,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,class_1_101,-1.180116e+42,4.721636e+26,-1.180116e+42,-1.180116e+42,-1.180116e+42,0.0,-1.180116e+42,-1.180116e+42,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,class_1_102,-1.180116e+42,4.721636e+26,-1.180116e+42,-1.180116e+42,-1.180116e+42,0.0,-1.180116e+42,-1.180116e+42,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31298,class_1_995,2.205004e+07,4.401704e+02,2.204931e+07,2.205076e+07,2.205004e+07,1450.0,2.204967e+07,2.205040e+07,725.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
31299,class_1_996,2.205153e+07,4.359440e+02,2.205081e+07,2.205225e+07,2.205154e+07,1440.0,2.205117e+07,2.205189e+07,715.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
31300,class_1_997,2.205300e+07,4.286369e+02,2.205230e+07,2.205371e+07,2.205301e+07,1410.0,2.205265e+07,2.205336e+07,705.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
31301,class_1_998,2.205445e+07,4.176847e+02,2.205375e+07,2.205513e+07,2.205445e+07,1380.0,2.205410e+07,2.205479e+07,685.1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [27]:
# Combine the class-0 windows with the class-1 windows into a single dataset.
final_data0_1= pd.concat([final_data0,final_data1]).reset_index(drop=True)

In [28]:
# Display the combined dataset.
final_data0_1

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,class_0_1,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,class_0_10,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,class_0_100,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,class_0_101,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,class_0_102,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48212,class_1_995,2.205004e+07,440.170422,22049312.0,22050762.0,22050037.0,1450.0,22049674.5,22050399.5,725.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
48213,class_1_996,2.205153e+07,435.943960,22050812.0,22052252.0,22051537.0,1440.0,22051174.5,22051889.5,715.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
48214,class_1_997,2.205300e+07,428.636893,22052301.6,22053711.6,22053007.0,1410.0,22052654.5,22053359.5,705.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
48215,class_1_998,2.205445e+07,417.684660,22053752.0,22055132.0,22054447.0,1380.0,22054104.4,22054789.5,685.1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [29]:
# Drop the window identifier: only features and labels are kept.
final_data0_1.drop(columns=['instance'], inplace=True)

In [30]:
# Export the combined class-0 / class-1 dataset.
# Save to CSV
final_data0_1.to_csv('final_data0_1.csv', index=False)

# Folder 2

In [33]:
# ============================================================
# 4.2 CLASS 2
# ============================================================
# List the Parquet files of folder 2.
folder = '/kaggle/input/3w-dataset/2.0.0/2'
file_names = os.listdir(folder)

In [34]:
# Extract windowed statistical features from the first 40 files of folder 2 and concatenate them.
rows = []
for i  in file_names[:40] : 
    data = features_engineering(folder,i)
    rows.append(data)
    
final_data2= pd.concat(rows)

In [35]:
# Remove windows labelled as normal operation (class 0) from the class-2 table.
final_data2 = final_data2[final_data2['class']!=0]

In [36]:
# Reset the row index after filtering.
final_data2.reset_index(drop= True, inplace = True)

In [37]:
# Display the class-2 feature table.
final_data2

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,class_2_1,2.963511e+07,801.046504,2.963421e+07,2.963679e+07,2.963490e+07,2579.099976,2.963440e+07,2.963568e+07,1274.274988,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
1,class_2_10,3.372114e+07,31198.957883,3.363237e+07,3.374494e+07,3.373238e+07,112566.387743,3.371757e+07,3.373973e+07,22163.472808,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
2,class_2_100,3.356206e+07,4086.003786,3.355347e+07,3.356707e+07,3.356274e+07,13601.796721,3.355845e+07,3.356541e+07,6952.299255,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
3,class_2_101,3.356354e+07,3077.513716,3.355758e+07,3.356859e+07,3.356294e+07,11014.298353,3.356148e+07,3.356624e+07,4759.673213,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
4,class_2_102,3.356455e+07,2683.704335,3.356004e+07,3.356852e+07,3.356430e+07,8478.898079,3.356225e+07,3.356673e+07,4473.849332,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17126,class_2_95,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
17127,class_2_96,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
17128,class_2_97,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
17129,class_2_98,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2


In [38]:
# Combine the class-0 windows with the class-2 windows into a single dataset.
final_data0_2= pd.concat([final_data0,final_data2]).reset_index(drop=True)

In [39]:
# Drop the window identifier: only features and labels are kept.
final_data0_2.drop(columns=['instance'], inplace=True)

In [40]:
# Export the combined class-0 / class-2 dataset.
# Save to CSV
final_data0_2.to_csv('final_data0_2.csv', index=False)

# Folder 3

In [41]:
# ============================================================
# 4.3 CLASS 3
# ============================================================
# List the Parquet files of folder 3.
folder = '/kaggle/input/3w-dataset/2.0.0/3'
file_names = os.listdir(folder)

In [42]:
# Extract windowed statistical features from the first 30 files of folder 3 and concatenate them.
rows = []
for i  in file_names[:30] : 
    data = features_engineering(folder,i)
    rows.append(data)
    
final_data3= pd.concat(rows)

In [43]:
# Remove windows labelled as normal operation (class 0) from the class-3 table.
final_data3 = final_data3[final_data3['class']!=0]

In [44]:
# Reset the row index after filtering.
final_data3.reset_index(drop= True, inplace = True)

In [45]:
# Display the class-3 feature table.
final_data3

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,class_3_1,2.907472e+07,8132.555616,2.906139e+07,2.908819e+07,2.907467e+07,26799.900001,2.906802e+07,2.908140e+07,13377.300003,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,3
1,class_3_10,2.927736e+07,5459.050971,2.926832e+07,2.928620e+07,2.927741e+07,17880.399995,2.927284e+07,2.928189e+07,9050.124998,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,3
2,class_3_100,2.938162e+07,689.233701,2.938050e+07,2.938293e+07,2.938154e+07,2430.199997,2.938109e+07,2.938213e+07,1042.874995,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,3
3,class_3_1000,2.345143e+07,20971.869434,2.341928e+07,2.348815e+07,2.345019e+07,68870.339185,2.343378e+07,2.346831e+07,34526.397519,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,3
4,class_3_1001,2.339564e+07,11808.661143,2.337869e+07,2.341740e+07,2.339430e+07,38705.042638,2.338559e+07,2.340494e+07,19343.589694,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44638,class_3_95,2.409320e+07,3569.451228,2.408705e+07,2.409864e+07,2.409372e+07,11590.000000,2.408986e+07,2.409581e+07,5952.500000,...,1.192053,0.055527,0.038283,1.156906,1.157857,40.218974,-0.191166,-0.482107,0.0,3
44639,class_3_96,2.410481e+07,3875.911773,2.409870e+07,2.411178e+07,2.410532e+07,13080.000000,2.410193e+07,2.410711e+07,5180.000000,...,1.140189,0.093571,0.086416,1.068330,1.073922,34.599235,-1.494151,1.104097,0.0,3
44640,class_3_97,2.411752e+07,3879.072410,2.411161e+07,2.412358e+07,2.411748e+07,11970.000000,2.411422e+07,2.412088e+07,6655.000000,...,0.790141,0.055720,0.053098,0.781133,0.784518,18.464031,1.531490,1.802220,0.0,3
44641,class_3_98,2.413057e+07,3763.443107,2.412443e+07,2.413669e+07,2.413064e+07,12260.000000,2.412706e+07,2.413420e+07,7145.000000,...,1.134819,0.073931,0.035792,1.098561,1.099279,36.252430,-0.304305,-1.334379,0.0,3


In [46]:
# Combine the class-0 windows with the class-3 windows into a single dataset.
final_data0_3= pd.concat([final_data0,final_data3]).reset_index(drop=True)

In [47]:
# Drop the window identifier: only features and labels are kept.
final_data0_3.drop(columns=['instance'], inplace=True)

In [48]:
# Export the combined class-0 / class-3 dataset.
# Save to CSV
final_data0_3.to_csv('final_data0_3.csv', index=False)

# Folder 4

In [55]:
# ============================================================
# 4.4 CLASS 4
# ============================================================
# List the Parquet files of folder 4.
folder = '/kaggle/input/3w-dataset/2.0.0/4'
file_names = os.listdir(folder)

In [56]:
# Extract windowed statistical features from the first 120 files of folder 4 and concatenate them.
rows = []
for i  in file_names[:120] : 
    data = features_engineering(folder,i)
    rows.append(data)
    
final_data4= pd.concat(rows)

In [57]:
# Remove windows labelled as normal operation (class 0) from the class-4 table.
final_data4 = final_data4[final_data4['class']!=0]

In [58]:
# Reset the row index after filtering.
final_data4.reset_index(drop= True, inplace = True)

In [59]:
# Display the class-4 feature table.
final_data4

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,class_4_1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4
1,class_4_10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4
2,class_4_100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4
3,class_4_101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4
4,class_4_102,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28642,class_4_95,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4
28643,class_4_96,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4
28644,class_4_97,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4
28645,class_4_98,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4


In [60]:
# Combine the class-0 windows with the class-4 windows into a single dataset.
final_data0_4= pd.concat([final_data0,final_data4]).reset_index(drop=True)

In [61]:
# Drop the window identifier: only features and labels are kept.
final_data0_4.drop(columns=['instance'], inplace=True)

In [62]:
# Export the combined class-0 / class-4 dataset.
# Save to CSV
final_data0_4.to_csv('final_data0_4.csv', index=False)

# Folder 5

In [68]:
# ============================================================
# 4.5 CLASS 5
# ============================================================
# List the Parquet files of folder 5.
folder = '/kaggle/input/3w-dataset/2.0.0/5'
file_names = os.listdir(folder)

In [69]:
# Extract windowed statistical features from the first 50 files of folder 5 and concatenate them.
rows = []
for i  in file_names[:50] : 
    data = features_engineering(folder,i)
    rows.append(data)
    
final_data5= pd.concat(rows)

In [70]:
# Remove windows labelled as normal operation (class 0) from the class-5 table.
final_data5 = final_data5[final_data5['class']!=0]

In [71]:
# Reset the row index after filtering.
final_data5.reset_index(drop= True, inplace = True)

In [72]:
# Display the class-5 feature table.
final_data5

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,class_5_1,3.221704e+07,782.287989,3.221541e+07,3.221817e+07,3.221717e+07,2760.925924,3.221648e+07,3.221773e+07,1246.859603,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5
1,class_5_10,3.218034e+07,1397.502668,3.217798e+07,3.218265e+07,3.218022e+07,4674.245136,3.217925e+07,3.218155e+07,2295.742426,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5
2,class_5_100,3.154938e+07,2743.259034,3.154471e+07,3.155372e+07,3.154928e+07,9014.848423,3.154697e+07,3.155162e+07,4643.570883,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5
3,class_5_101,3.154018e+07,2691.110652,3.153581e+07,3.154462e+07,3.154028e+07,8811.842927,3.153802e+07,3.154245e+07,4426.761752,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5
4,class_5_102,3.153103e+07,2694.182996,3.152658e+07,3.153546e+07,3.153110e+07,8878.596001,3.152883e+07,3.153339e+07,4556.545324,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45576,class_5_96,2.136772e+07,8886.457940,2.135306e+07,2.138233e+07,2.136773e+07,29269.684080,2.136040e+07,2.137504e+07,14637.707553,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5
45577,class_5_960,2.064139e+07,0.000000,2.064139e+07,2.064139e+07,2.064139e+07,0.000000,2.064139e+07,2.064139e+07,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5
45578,class_5_97,2.133729e+07,8968.166402,2.132251e+07,2.135205e+07,2.133730e+07,29539.158880,2.132992e+07,2.134468e+07,14762.859339,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5
45579,class_5_98,2.130664e+07,9024.669638,2.129176e+07,2.132149e+07,2.130665e+07,29729.263277,2.129921e+07,2.131409e+07,14877.228606,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5


In [73]:
# Combine the class-0 windows with the class-5 windows into a single dataset.
final_data0_5= pd.concat([final_data0,final_data5]).reset_index(drop=True)

In [74]:
# Drop the window identifier: only features and labels are kept.
final_data0_5.drop(columns=['instance'], inplace=True)

In [75]:
# Export the combined class-0 / class-5 dataset.
# Save to CSV
final_data0_5.to_csv('final_data0_5.csv', index=False)

# Folder 6 

In [81]:
# ============================================================
# 4.6 CLASS 6
# ============================================================
# List the Parquet files of folder 6.
folder = '/kaggle/input/3w-dataset/2.0.0/6'
file_names = os.listdir(folder)

In [82]:
# Extract windowed statistical features from the first 40 files of folder 6 and concatenate them.
rows = []
for i  in file_names[:40] : 
    data = features_engineering(folder,i)
    rows.append(data)
    
final_data6= pd.concat(rows)

In [83]:
# Remove windows labelled as normal operation (class 0) from the class-6 table.
final_data6 = final_data6[final_data6['class']!=0]

In [84]:
# Reset the row index after filtering.
final_data6.reset_index(drop= True, inplace = True)

In [85]:
# Display the class-6 feature table.
final_data6

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,class_6_1,2.064269e+07,19.979163,2.064266e+07,2.064274e+07,2.064270e+07,77.643595,2.064268e+07,2.064271e+07,28.806319,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6
1,class_6_10,2.069520e+07,4278.235231,2.068821e+07,2.070233e+07,2.069513e+07,14120.315801,2.069173e+07,2.069867e+07,6945.149122,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6
2,class_6_100,2.123246e+07,368.701910,2.123195e+07,2.123309e+07,2.123241e+07,1146.670563,2.123215e+07,2.123278e+07,637.531914,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6
3,class_6_101,2.123360e+07,353.840036,2.123312e+07,2.123422e+07,2.123355e+07,1098.286795,2.123330e+07,2.123390e+07,600.704297,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6
4,class_6_102,2.123471e+07,339.282901,2.123425e+07,2.123531e+07,2.123466e+07,1059.663159,2.123441e+07,2.123498e+07,566.500182,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33595,class_6_95,2.242372e+07,2066.192626,2.242030e+07,2.242709e+07,2.242374e+07,6797.281549,2.242202e+07,2.242543e+07,3409.126192,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6
33596,class_6_96,2.243060e+07,1976.033430,2.242733e+07,2.243384e+07,2.243061e+07,6512.718427,2.242898e+07,2.243223e+07,3246.286397,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6
33597,class_6_97,2.243722e+07,1908.398733,2.243406e+07,2.244035e+07,2.243723e+07,6286.310684,2.243565e+07,2.243880e+07,3146.165028,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6
33598,class_6_98,2.244361e+07,1840.764550,2.244056e+07,2.244663e+07,2.244362e+07,6068.640729,2.244209e+07,2.244513e+07,3036.504845,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6


In [86]:
# Combine the class-0 windows with the class-6 windows into a single dataset.
final_data0_6= pd.concat([final_data0,final_data6]).reset_index(drop=True)

In [87]:
# Drop the window identifier: only features and labels are kept.
final_data0_6.drop(columns=['instance'], inplace=True)

In [88]:
# Export the combined class-0 / class-6 dataset.
# Save to CSV
final_data0_6.to_csv('final_data0_6.csv', index=False)

# Folder 7

In [89]:
# ============================================================
# 4.7 CLASS 7
# ============================================================
# List the Parquet files of folder 7.
folder = '/kaggle/input/3w-dataset/2.0.0/7'
file_names = os.listdir(folder)

In [90]:
# Extract windowed statistical features from the first 8 files of folder 7 and concatenate them.
rows = []
for i  in file_names[:8] : 
    data = features_engineering(folder,i)
    rows.append(data)
    
final_data7= pd.concat(rows)

In [91]:
# Remove windows labelled as normal operation (class 0) from the class-7 table.
final_data7 = final_data7[final_data7['class']!=0]

In [92]:
# Reset the row index after filtering.
final_data7.reset_index(drop= True, inplace = True)

In [93]:
# Display the class-7 feature table.
final_data7

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,class_7_1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,7
1,class_7_10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,7
2,class_7_100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,7
3,class_7_1000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,7
4,class_7_1001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45850,class_7_9995,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.930659,0.001270,0.000657,1.930024,1.930024,111.749820,0.000263,-1.200282,0.0,7
45851,class_7_9996,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.931545,0.002974,0.001391,1.929720,1.929720,111.714634,0.715806,-1.406609,0.0,7
45852,class_7_9997,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.932554,0.002095,0.001613,1.931126,1.931127,111.877528,-0.397275,-0.314419,0.0,7
45853,class_7_9998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.930384,0.002701,0.001899,1.929063,1.929064,111.638683,0.333698,-0.437418,0.0,7


In [94]:
# Combine the class-0 windows with the class-7 windows into a single dataset.
final_data0_7= pd.concat([final_data0,final_data7]).reset_index(drop=True)

In [95]:
# Drop the window identifier: only features and labels are kept.
final_data0_7.drop(columns=['instance'], inplace=True)

In [96]:
# Export the combined class-0 / class-7 dataset.
# Save to CSV
final_data0_7.to_csv('final_data0_7.csv', index=False)

# Folder 8

In [97]:
# ============================================================
# 4.8 CLASS 8
# ============================================================
# List the Parquet files of folder 8.
folder = '/kaggle/input/3w-dataset/2.0.0/8'
file_names = os.listdir(folder)

In [98]:
# Extract windowed statistical features from the first 20 files of folder 8 and concatenate them.
rows = []
for i  in file_names[:20] : 
    data = features_engineering(folder,i)
    rows.append(data)
    
final_data8= pd.concat(rows)

In [99]:
# Remove windows labelled as normal operation (class 0) from the class-8 table.
final_data8 = final_data8[final_data8['class']!=0]

In [100]:
# Reset the row index after filtering.
final_data8.reset_index(drop= True, inplace = True)

In [101]:
# Display the class-8 feature table.
final_data8

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,class_8_1,2.756782e+07,4.475646e+02,2.756722e+07,2.756861e+07,2.756777e+07,1.393851e+03,2.756741e+07,2.756820e+07,7.972703e+02,...,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,8
1,class_8_10,2.758559e+07,6.889908e+02,2.758443e+07,2.758675e+07,2.758558e+07,2.321972e+03,2.758503e+07,2.758614e+07,1.114788e+03,...,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,8
2,class_8_100,3.372144e+07,1.134450e+03,3.371929e+07,3.372396e+07,3.372155e+07,4.670000e+03,3.372056e+07,3.372205e+07,1.483700e+03,...,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,8
3,class_8_101,3.372419e+07,1.611921e+03,3.372092e+07,3.372922e+07,3.372374e+07,8.294800e+03,3.372308e+07,3.372520e+07,2.122600e+03,...,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,8
4,class_8_102,3.372555e+07,1.310898e+03,3.372313e+07,3.372927e+07,3.372549e+07,6.139600e+03,3.372481e+07,3.372616e+07,1.344600e+03,...,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35689,class_8_995,1.563572e+07,1.152387e+06,1.323075e+07,1.651053e+07,1.650996e+07,3.279780e+06,1.475343e+07,1.651024e+07,1.756808e+06,...,0.047793,4.350000e-07,2.276667e-07,0.047793,0.047793,0.068525,0.000000,0.000000,0.0,8
35690,class_8_996,1.343404e+07,2.493768e+06,9.482426e+06,1.651134e+07,1.299668e+07,7.028914e+06,1.135662e+07,1.639379e+07,5.037172e+06,...,0.047792,3.550000e-07,1.885333e-07,0.047792,0.047792,0.068523,0.000000,0.000000,0.0,8
35691,class_8_997,1.651171e+07,1.188842e+02,1.651140e+07,1.651195e+07,1.651172e+07,5.500000e+02,1.651165e+07,1.651178e+07,1.250000e+02,...,0.047795,1.767500e-06,9.130000e-07,0.047794,0.047794,0.068529,0.000000,0.000000,0.0,8
35692,class_8_998,1.607520e+07,9.281560e+05,1.323345e+07,1.651262e+07,1.651233e+07,3.279170e+06,1.651200e+07,1.651248e+07,4.725000e+02,...,0.047799,1.767500e-06,1.150586e-04,0.047873,0.047873,0.068755,2.089242,3.282374,0.0,8


In [102]:
# Combine the class-0 windows with the class-8 windows into a single dataset.
final_data0_8= pd.concat([final_data0,final_data8]).reset_index(drop=True)

In [103]:
# Drop the window identifier: only features and labels are kept.
final_data0_8.drop(columns=['instance'], inplace=True)

In [104]:
# Export the combined class-0 / class-8 dataset.
# Save to CSV
final_data0_8.to_csv('final_data0_8.csv', index=False)

### Exploratory inspection of a raw file (class 2)

In [4]:
# Exploratory check: load one raw class-2 Parquet file for inspection.
test = pd.read_parquet('/kaggle/input/3w-dataset/2.0.0/2/SIMULATED_00001.parquet')

In [5]:
# Display the raw file (all 27 sensor/state columns plus `class` and `state`).
test

,ABER-CKGL,ABER-CKP,ESTADO-DHSV,ESTADO-M1,ESTADO-M2,ESTADO-PXO,ESTADO-SDV-GL,ESTADO-SDV-P,ESTADO-W1,ESTADO-W2,...,PT-P,P-TPT,QBS,QGL,T-JUS-CKP,T-MON-CKP,T-PDG,T-TPT,class,state
timestamp,,,,,,,,,,,,,,,,,,,,,
2018-05-13 01:53:17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,20228170.00,NaN,NaN,90.817520,NaN,NaN,124.573200,0,0
2018-05-13 01:53:18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,20228170.00,NaN,NaN,90.817520,NaN,NaN,124.573200,0,0
2018-05-13 01:53:19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,20228170.00,NaN,NaN,90.817529,NaN,NaN,124.573200,0,0
2018-05-13 01:53:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,20228170.00,NaN,NaN,90.817530,NaN,NaN,124.573200,0,0
2018-05-13 01:53:21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,20228170.00,NaN,NaN,90.817530,NaN,NaN,124.573200,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-05-13 09:53:11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,5991254.36,NaN,NaN,29.271505,NaN,NaN,3.426753,2,0
2018-05-13 09:53:12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,5991477.68,NaN,NaN,29.244672,NaN,NaN,3.426746,2,0
2018-05-13 09:53:13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,5991689.34,NaN,NaN,29.224872,NaN,NaN,3.426739,2,0


In [10]:
# List the columns that contain at least one missing value.
num_columns = test.columns[test.isna().any()].tolist()
num_columns 

['ABER-CKGL',
 'ABER-CKP',
 'ESTADO-DHSV',
 'ESTADO-M1',
 'ESTADO-M2',
 'ESTADO-PXO',
 'ESTADO-SDV-GL',
 'ESTADO-SDV-P',
 'ESTADO-W1',
 'ESTADO-W2',
 'ESTADO-XO',
 'P-ANULAR',
 'P-JUS-BS',
 'P-JUS-CKGL',
 'P-JUS-CKP',
 'P-MON-CKGL',
 'P-MON-SDV-P',
 'PT-P',
 'QBS',
 'QGL',
 'T-MON-CKP',
 'T-PDG']

In [16]:
# Total number of columns in the raw file.
len(test.columns)

29

# Folder 9

In [80]:
# ============================================================
# 4.9 CLASS 9
# ============================================================
# List the Parquet files of folder 9.
folder = '/kaggle/input/3w-dataset/2.0.0/9'
file_names = os.listdir(folder)

In [5]:
# Exploratory check: process a single class-9 file.
t = features_engineering(folder,file_names[8])

In [6]:
# Display the features extracted from that file.
t

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,class_9_1,2.508966e+07,346.882504,25089090.0,25090230.0,25089660.0,1140.0,25089370.0,25089950.0,580.0,...,0.647934,0.006013,0.003099,0.644991,0.645001,12.480799,0.012162,-1.189302,0.0,9
1,class_9_10,2.510146e+07,219.656210,25101110.0,25101820.0,25101460.0,710.0,25101275.0,25101635.0,360.0,...,0.639221,0.009799,0.005501,0.634134,0.634166,12.065010,-0.737692,-0.693863,0.0,9
2,class_9_100,2.503150e+07,49.879164,25031390.0,25031570.0,25031505.0,180.0,25031470.0,25031530.0,60.0,...,0.464559,0.021301,0.010698,0.454429,0.454590,6.199549,0.480244,-1.217514,0.0,9
3,class_9_101,2.503120e+07,106.427894,25031020.0,25031370.0,25031195.0,350.0,25031112.5,25031287.5,175.0,...,0.471740,0.021354,0.010076,0.461817,0.461957,6.402125,0.194044,-1.401838,0.0,9
4,class_9_102,2.503090e+07,47.323781,25030840.0,25031010.0,25030890.0,170.0,25030870.0,25030925.0,55.0,...,0.451530,0.002386,0.001334,0.450264,0.450266,6.082194,-0.294673,-0.694357,0.0,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273,class_9_95,2.503165e+07,106.527747,25031470.0,25031830.0,25031650.0,360.0,25031562.5,25031737.5,175.0,...,0.456030,0.004549,0.002353,0.453755,0.453763,6.177034,-0.000002,-1.199996,0.0,9
274,class_9_96,2.503133e+07,64.998674,25031260.0,25031460.0,25031290.0,200.0,25031272.5,25031377.5,105.0,...,0.460736,0.012303,0.007033,0.454355,0.454427,6.195122,-0.905205,-0.589345,0.0,9
275,class_9_97,2.503122e+07,19.815239,25031190.0,25031260.0,25031220.0,70.0,25031210.0,25031240.0,30.0,...,0.443873,0.013521,0.007025,0.437249,0.437324,5.737578,0.355727,-1.104252,0.0,9
276,class_9_98,2.503119e+07,29.635719,25031160.0,25031260.0,25031180.0,100.0,25031170.0,25031207.5,37.5,...,0.460862,0.004730,0.002517,0.458482,0.458491,6.306412,-0.420740,-1.059337,0.0,9


In [81]:
# Extract windowed statistical features from the first 7 files of folder 9 and concatenate them.
rows = []
for i  in file_names[:7] : 
    data = features_engineering(folder,i)
    rows.append(data)
    
final_data9= pd.concat(rows)

In [82]:
# Remove windows labelled as normal operation (class 0) from the class-9 table.
final_data9 = final_data9[final_data9['class']!=0]

In [83]:
# Reset the row index after filtering.
final_data9.reset_index(drop= True, inplace = True)

In [85]:
# Display the class-9 feature table.
final_data9

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,P-PDG,P-TPT,T-TPT,P-MON-CKP,T-JUS-CKP,P-JUS-CKGL,T-MON-CKP,QGL,class_x,class_y
0,class_9_1,2.219030e+07,6.933596,2.219028e+07,2.219030e+07,2.219030e+07,20.000000,2.219029e+07,2.219030e+07,10.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
1,class_9_10,2.218907e+07,42.314236,2.218900e+07,2.218914e+07,2.218907e+07,140.000000,2.218903e+07,2.218910e+07,68.675000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
2,class_9_100,2.218591e+07,4.926192,2.218591e+07,2.218592e+07,2.218591e+07,10.000000,2.218591e+07,2.218592e+07,10.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
3,class_9_1000,2.226668e+07,1565.471340,2.226410e+07,2.226983e+07,2.226741e+07,5730.688328,2.226514e+07,2.226796e+07,2823.613075,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
4,class_9_1001,2.226674e+07,1397.303989,2.226524e+07,2.226819e+07,2.226677e+07,2953.088445,2.226536e+07,2.226810e+07,2736.788440,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9235,class_9_995,2.989887e+07,4988.485167,2.988846e+07,2.990375e+07,2.990099e+07,15284.000000,2.989552e+07,2.990320e+07,7683.500000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
9236,class_9_996,2.989913e+07,2365.865476,2.989541e+07,2.990323e+07,2.989929e+07,7824.000000,2.989726e+07,2.990158e+07,4318.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
9237,class_9_997,2.989330e+07,1444.586273,2.989050e+07,2.989592e+07,2.989328e+07,5412.000000,2.989218e+07,2.989434e+07,2163.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
9238,class_9_998,2.988853e+07,1509.390194,2.988611e+07,2.989101e+07,2.988830e+07,4892.000000,2.988708e+07,2.988962e+07,2540.500000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>


In [7]:
# The class-0 filter and index reset are repeated (no effect on the result).
final_data9 = final_data9[final_data9['class']!=0]

In [8]:
# Reset the row index again.
final_data9.reset_index(drop= True, inplace = True)

In [9]:
# Display the class-9 feature table.
final_data9

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,P-PDG,P-TPT,T-TPT,P-MON-CKP,T-JUS-CKP,P-JUS-CKGL,T-MON-CKP,QGL,class_x,class_y
0,class_9_1,2.219030e+07,6.933596,2.219028e+07,2.219030e+07,2.219030e+07,20.000000,2.219029e+07,2.219030e+07,10.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
1,class_9_10,2.218907e+07,42.314236,2.218900e+07,2.218914e+07,2.218907e+07,140.000000,2.218903e+07,2.218910e+07,68.675000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
2,class_9_100,2.218591e+07,4.926192,2.218591e+07,2.218592e+07,2.218591e+07,10.000000,2.218591e+07,2.218592e+07,10.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
3,class_9_1000,2.226668e+07,1565.471340,2.226410e+07,2.226983e+07,2.226741e+07,5730.688328,2.226514e+07,2.226796e+07,2823.613075,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
4,class_9_1001,2.226674e+07,1397.303989,2.226524e+07,2.226819e+07,2.226677e+07,2953.088445,2.226536e+07,2.226810e+07,2736.788440,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7075,class_9_995,1.865171e+07,1110.701751,1.864989e+07,1.865347e+07,1.865173e+07,3580.000000,1.865077e+07,1.865265e+07,1878.200001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
7076,class_9_996,1.864834e+07,835.411791,1.864703e+07,1.864978e+07,1.864830e+07,2747.899996,1.864764e+07,1.864901e+07,1378.774998,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
7077,class_9_997,1.864604e+07,497.527045,1.864533e+07,1.864696e+07,1.864599e+07,1627.199995,1.864562e+07,1.864644e+07,818.074996,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
7078,class_9_998,1.864494e+07,193.058972,1.864464e+07,1.864530e+07,1.864492e+07,658.599997,1.864478e+07,1.864509e+07,306.974999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>


In [15]:
# Exploratory check: load one raw class-9 Parquet file.
test = pd.read_parquet('/kaggle/input/3w-dataset/2.0.0/9/SIMULATED_00001.parquet')

In [16]:
# Display the raw class-9 file.
test

,ABER-CKGL,ABER-CKP,ESTADO-DHSV,ESTADO-M1,ESTADO-M2,ESTADO-PXO,ESTADO-SDV-GL,ESTADO-SDV-P,ESTADO-W1,ESTADO-W2,...,PT-P,P-TPT,QBS,QGL,T-JUS-CKP,T-MON-CKP,T-PDG,T-TPT,class,state
timestamp,,,,,,,,,,,,,,,,,,,,,
2018-09-29 02:34:05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9.443209e+06,NaN,2.687037,78.756740,NaN,NaN,93.677530,0,0
2018-09-29 02:34:06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9.443208e+06,NaN,2.687038,78.756740,NaN,NaN,93.677515,0,0
2018-09-29 02:34:07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9.443198e+06,NaN,2.687039,78.756740,NaN,NaN,93.677503,0,0
2018-09-29 02:34:08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9.443167e+06,NaN,2.687040,78.756740,NaN,NaN,93.677478,0,0
2018-09-29 02:34:09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9.443119e+06,NaN,2.687041,78.756740,NaN,NaN,93.677448,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-09-29 11:13:59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1.655428e+07,NaN,0.001790,26.492992,NaN,NaN,79.711839,9,0
2018-09-29 11:14:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1.655446e+07,NaN,0.001790,26.486364,NaN,NaN,79.710919,9,0
2018-09-29 11:14:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1.655467e+07,NaN,0.001790,26.479499,NaN,NaN,79.710005,9,0


In [22]:
# Inspect the rows carrying the transient label 109 (transient phase of class 9).
test[test['class']==109]

,ABER-CKGL,ABER-CKP,ESTADO-DHSV,ESTADO-M1,ESTADO-M2,ESTADO-PXO,ESTADO-SDV-GL,ESTADO-SDV-P,ESTADO-W1,ESTADO-W2,...,PT-P,P-TPT,QBS,QGL,T-JUS-CKP,T-MON-CKP,T-PDG,T-TPT,class,state
timestamp,,,,,,,,,,,,,,,,,,,,,
2018-09-29 04:14:05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9439846.06,NaN,2.692655,78.741093,NaN,NaN,93.633837,109,0
2018-09-29 04:14:06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9439888.00,NaN,2.692654,78.741090,NaN,NaN,93.633847,109,0
2018-09-29 04:14:07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9439936.52,NaN,2.692649,78.741090,NaN,NaN,93.633865,109,0
2018-09-29 04:14:08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9439963.06,NaN,2.692639,78.741083,NaN,NaN,93.633870,109,0
2018-09-29 04:14:09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9439960.60,NaN,2.692624,78.741080,NaN,NaN,93.633855,109,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-09-29 05:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9187915.24,NaN,0.123690,78.571280,NaN,NaN,94.620861,109,0
2018-09-29 05:20:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9187691.98,NaN,0.121704,78.571042,NaN,NaN,94.621971,109,0
2018-09-29 05:20:02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9187468.24,NaN,0.116674,78.570802,NaN,NaN,94.623089,109,0


In [13]:
# Display the class-9 feature table.
final_data9

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,P-PDG,P-TPT,T-TPT,P-MON-CKP,T-JUS-CKP,P-JUS-CKGL,T-MON-CKP,QGL,class_x,class_y
0,class_9_1,2.219030e+07,6.933596,2.219028e+07,2.219030e+07,2.219030e+07,20.000000,2.219029e+07,2.219030e+07,10.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
1,class_9_10,2.218907e+07,42.314236,2.218900e+07,2.218914e+07,2.218907e+07,140.000000,2.218903e+07,2.218910e+07,68.675000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
2,class_9_100,2.218591e+07,4.926192,2.218591e+07,2.218592e+07,2.218591e+07,10.000000,2.218591e+07,2.218592e+07,10.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
3,class_9_1000,2.226668e+07,1565.471340,2.226410e+07,2.226983e+07,2.226741e+07,5730.688328,2.226514e+07,2.226796e+07,2823.613075,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
4,class_9_1001,2.226674e+07,1397.303989,2.226524e+07,2.226819e+07,2.226677e+07,2953.088445,2.226536e+07,2.226810e+07,2736.788440,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15513,class_9_995,1.938147e+07,644.519551,1.938040e+07,1.938252e+07,1.938148e+07,2120.000000,1.938094e+07,1.938200e+07,1063.850000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
15514,class_9_996,1.938360e+07,608.613022,1.938259e+07,1.938460e+07,1.938361e+07,2005.400000,1.938311e+07,1.938410e+07,995.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
15515,class_9_997,1.938562e+07,578.692673,1.938466e+07,1.938657e+07,1.938563e+07,1905.400000,1.938515e+07,1.938610e+07,952.700000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
15516,class_9_998,1.938755e+07,553.461283,1.938663e+07,1.938846e+07,1.938756e+07,1825.400000,1.938710e+07,1.938801e+07,912.700000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>


In [9]:
# Combine the class-0 windows with the class-9 windows into a single dataset.
final_data0_9= pd.concat([final_data0,final_data9]).reset_index(drop=True)

NameError: name 'final_data0' is not defined

In [79]:
# Drop the window identifier: only features and labels are kept.
final_data0_9.drop(columns=['instance'], inplace=True)

In [80]:
# Export the combined class-0 / class-9 dataset.
# Note: the experiments use the class-9 file regenerated in notebook 01b.
# Save to CSV
final_data0_9.to_csv('final_data0_9.csv', index=False)

*The remaining empty cells are kept from the original notebook.*